In [27]:
import pandas as pd 


In [28]:
import numpy as np
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import nltk 
from nltk.corpus import stopwords
nltk.download('stopwords')
nltk.download('punkt')  


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/anurag_77y/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     /Users/anurag_77y/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [29]:
df = pd.read_csv('spam.csv')

In [30]:
df.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


In [31]:
keep_cols = ['v1', 'v2']
df = df[keep_cols]

In [32]:
df.head()

,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [33]:
df.rename(columns = {'v1': 'target', 'v2': 'text'}, inplace = True)
df.head()

,target,text
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [34]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['target'] = le.fit_transform(df['target'])
df.sample(10)

,target,text
1308,0,I jokin oni lar.. ÌÏ busy then i wun disturb Ì_.
4280,0,"Wn u r hurt by d prsn who s close 2 u, do figh..."
277,0,"Awesome, I'll see you in a bit"
145,0,Whats the staff name who is taking class for us?
2545,0,So are you guys asking that i get that slipper...
5056,0,Hey next sun 1030 there's a basic yoga course....
1696,0,"Sorry man, my stash ran dry last night and I c..."
950,0,"Awesome, lemme know whenever you're around"
4124,0,May b approve panalam...but it should have mor...
3183,0,Good morning pookie pie! Lol hope I didn't wak...


In [35]:
df.duplicated().sum()

np.int64(403)

In [36]:
len(df)

5572

In [37]:
df = df.drop_duplicates(keep='first')
len(df)

5169

In [38]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()
import string

In [39]:
def transform_text(text):
    text= text.lower()
    text = nltk.word_tokenize(text)
    y=[]
    for i in text:
        if i.isalnum():
            y.append(i)
    text = y[:]
    y.clear()
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)
    text = y[:]
    y.clear()
    for i in text:
        y.append(ps.stem(i))
    return " ".join(y)

In [40]:
df['transformed_text'] = df['text'].apply(transform_text)
df.head()

,target,text,transformed_text
0,0,"Go until jurong point, crazy.. Available only ...",go jurong point crazi avail bugi n great world...
1,0,Ok lar... Joking wif u oni...,ok lar joke wif u oni
2,1,Free entry in 2 a wkly comp to win FA Cup fina...,free entri 2 wkli comp win fa cup final tkt 21...
3,0,U dun say so early hor... U c already then say...,u dun say earli hor u c alreadi say
4,0,"Nah I don't think he goes to usf, he lives aro...",nah think goe usf live around though


In [41]:
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
tfid = TfidfVectorizer(max_features=500)

In [42]:
X = tfid.fit_transform(df['transformed_text']).toarray()
y = df['target'].values

In [43]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=2)

In [44]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier

In [45]:
svc = SVC(kernel= "sigmoid", gamma  = 1.0)
knc = KNeighborsClassifier()
mnb = MultinomialNB()
dtc = DecisionTreeClassifier(max_depth = 5)
lrc = LogisticRegression(solver = 'liblinear', penalty = 'l1')
rfc = RandomForestClassifier(n_estimators = 50, random_state = 2 )
abc = AdaBoostClassifier(n_estimators = 50, random_state = 2)
bc = BaggingClassifier(n_estimators = 50, random_state = 2)
etc = ExtraTreesClassifier(n_estimators = 50, random_state = 2)
gbdt = GradientBoostingClassifier(n_estimators = 50, random_state = 2)    
xgb  = XGBClassifier(n_estimators = 50, random_state = 2)

In [46]:
dic = {
    'SVC': svc,
    'KNC': knc,
    'MNB': mnb,
    'DTC': dtc, 
    'LR': lrc,
    'RFC': rfc,
    'ABC': abc,
    'BC': bc,
    'ETC': etc,
    'GBDT': gbdt,
    'XGB': xgb
}

In [47]:
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix,classification_report

for name,model in dic.items():
    clf = model.fit(X_train,y_train)
    y_pred = clf.predict(X_test)
    print()
    print(f"Model: {name}")
    print(f"Accuracy Score: {accuracy_score(y_test,y_pred)}")
    print(f"Precision Score: {precision_score(y_test,y_pred)}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_test,y_pred)}")
    print(f"Classification Report:\n{classification_report(y_test,y_pred)}")



Model: SVC
Accuracy Score: 0.9661508704061895
Precision Score: 0.9327731092436975
Confusion Matrix:
[[888   8]
 [ 27 111]]
Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       896
           1       0.93      0.80      0.86       138

    accuracy                           0.97      1034
   macro avg       0.95      0.90      0.92      1034
weighted avg       0.97      0.97      0.97      1034


Model: KNC
Accuracy Score: 0.9274661508704062
Precision Score: 1.0
Confusion Matrix:
[[896   0]
 [ 75  63]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      1.00      0.96       896
           1       1.00      0.46      0.63       138

    accuracy                           0.93      1034
   macro avg       0.96      0.73      0.79      1034
weighted avg       0.93      0.93      0.92      1034


Model: MNB
Accuracy Score: 0.9709864603481625
Precision Score: 0.9